# 10 - End-to-end case study: MiniSearch is slow


## Goal

Take a realistic user symptom and walk every layer of the lab: seed-term extraction, traceability, components, log category, config key, D-Bus method, tests. End with a well-formed answer plus cited evidence.


## Prerequisites

- All previous notebooks (01-09) read. This is the flagship learning notebook.
- Mental shift: stop thinking about subsystems in isolation and start tracing a single symptom across all of them.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. The symptom

> *"MiniSearch is slow when opening folders with many files."*

This is the same symptom the eval set targets and the same one embedded in the synthetic logs. We will trace it from English to a citable, evidence-backed answer.


In [ ]:
SYMPTOM = 'MiniSearch is slow when opening folders with many files'
print(SYMPTOM)


## 2. Build the graph


In [ ]:
from src.common.paths import MINI_REPO
from src.repo_ingest.scanner import scan
from src.repo_ingest.cmake_reader import read_cmake
from src.repo_ingest.cpp_reader import read_cpp
from src.repo_ingest.qml_reader import read_qml
from src.repo_ingest.dbus_reader import read_dbus
from src.repo_ingest.kconfig_reader import read_kconfig
from src.repo_ingest.desktop_file_reader import read_desktop
from src.repo_ingest.log_reader import read_log
from src.ontology.extractor import (
    ExtractionBundle, from_cmake, from_cpp, from_qml, from_dbus,
    from_kconfig, from_desktop, from_log,
)
from src.ontology.schema import Entity
from src.common.ids import make_id
from src.graph.builder import build_graph

rep = scan(MINI_REPO)
b = ExtractionBundle()
rid = b.add_entity(Entity(id=make_id('Repository', MINI_REPO.name),
                          type='Repository', name=MINI_REPO.name,
                          source_path=str(MINI_REPO)))
for sf in rep.by_kind('cmake'): from_cmake(b, read_cmake(sf.path), rid)
for sf in rep.by_kind('cpp_header') + rep.by_kind('cpp_source'): from_cpp(b, read_cpp(sf.path))
for sf in rep.by_kind('qml'): from_qml(b, read_qml(sf.path))
for sf in rep.by_kind('dbus'): from_dbus(b, read_dbus(sf.path))
for sf in rep.by_kind('kconfig'): from_kconfig(b, read_kconfig(sf.path))
for sf in rep.by_kind('desktop'): from_desktop(b, read_desktop(sf.path))
for sf in rep.by_kind('log'): from_log(b, read_log(sf.path))
g = build_graph(b)


## 3. Step 1 - Seed terms

`_seed_terms` (in `src/traceability/symptom_to_code.py`) extracts CamelCase tokens and KDE-shaped keywords from the symptom string. The terms it pulls out are the starting set for graph lookup.


In [ ]:
from src.traceability.symptom_to_code import _seed_terms

seeds = _seed_terms(SYMPTOM)
print('seed terms:', seeds)


## 4. Step 2 - Traceability

`trace(g, symptom)` matches the seed terms against entity names and expands outward along high-signal relations (`EMITS`, `READS_CONFIG`, `LOGS_TO`, `EXPOSES_DBUS`, etc.). The result is a ranked evidence list.


In [ ]:
from src.traceability.symptom_to_code import trace

tr = trace(g, SYMPTOM, k=4)
print(f'evidence items: {len(tr.evidence)}')
for ev in tr.evidence[:12]:
    print(f'  conf={ev.confidence:.2f}  {ev.type:14s} {ev.name:30s}'
          f'  [{ev.reason}]')


## 5. Step 3 - Components

Filter the evidence down to actual C++ classes and QML components. These are the *places* the bug most likely lives.


In [ ]:
components = [e for e in tr.evidence if e.type in {'CppClass', 'QmlComponent'}]
for c in components:
    print(f'{c.type:14s} {c.name:30s} at {c.source_path}:{c.source_line}')


## 6. Step 4 - Log category

Which logging channel emits messages from these components? Enable that one to see live traces.


In [ ]:
log_cats = [e for e in tr.evidence if e.type == 'LogCategory']
for c in log_cats:
    print(f'enable: QT_LOGGING_RULES="{c.name}.debug=true"  (source: {c.source_path})')


## 7. Step 5 - Config key

Which user-tunable knob is the most likely throttle?


In [ ]:
config_keys = [e for e in tr.evidence if e.type == 'ConfigKey']
for k in config_keys:
    node = g.nodes.get(k.entity_id, {})
    print(f"{k.name:18s} type={node.get('prop_type','?'):8s}"
          f" default={node.get('prop_default','?')}"
          f" group={node.get('prop_group','?')}")


## 8. Step 6 - D-Bus method

Which D-Bus method is the entry point from outside processes? You can call it from the shell with `qdbus` to reproduce the symptom.


In [ ]:
dbus_methods = [e for e in tr.evidence if e.type == 'DbusMethod']
for m in dbus_methods:
    node = g.nodes.get(m.entity_id, {})
    iface = node.get('prop_interface', '?')
    print(f'qdbus {iface} /<obj> {m.name}')


## 9. Step 7 - Tests

The mini repo ships one `tst_kfilesearcher.cpp` under `tests/`. The scanner indexed it as a `SourceFile`; in a richer ontology each test case would be its own entity. For now we look for source files under a `tests/` path.


In [ ]:
test_files = [n for n, d in g.nodes(data=True)
              if d.get('type') in {'SourceFile', 'HeaderFile'}
              and 'tests' in (d.get('source_path') or '').lower()]
for t in test_files:
    d = g.nodes[t]
    print(f"test: {d.get('name')}  {d.get('source_path')}")


## 10. The cited answer

`answer(g, SYMPTOM)` runs the same retriever (`src/rag/graph_retriever.py`) we have been walking by hand, then assembles a Markdown answer with inline citations.


In [ ]:
from src.rag.answer_with_evidence import answer

a = answer(g, SYMPTOM, k=8)
try:
    from IPython.display import Markdown, display
    display(Markdown(a.text))
except ImportError:
    print(a.text)


## 11. Reading the answer

Notice that every claim in the answer maps back to an `Evidence` line with `file:line` coordinates. That property is the whole point of an evidence-grounded lab: the model can be **wrong**, but it cannot be **vague**.

If a teammate disputes the answer, they have a finite list of source lines to read.


## Summary

You walked a real symptom through every stage of the pipeline and ended with a citation-rich answer. This is the workflow the fine-tuned KDE SLM is meant to automate. Notebook 08's eval suite is how you verify it does.


## Exercises

1. Run the same notebook with a different symptom (e.g. *'MiniSearch hangs when cancelling a search'*). Where does the trace go right or wrong?
2. For each evidence item, count how many SFT examples in `artifacts/datasets/mini_repo_sft_v0.jsonl` cite the same entity. Are there gaps?
3. Propose one new relation type (e.g. `THROTTLED_BY`) that would make the answer to the slow-folders symptom shorter and more direct.
